In [53]:
import pandas as pd
import numpy as np
import re

# Load Datasets

In [54]:
playlist = pd.read_csv("dataset/playlist_with_segment.csv")
track = pd.read_csv("dataset/tracks_new.csv")

In [55]:
def convert_string_array_to_list(s):
    """Convert a string representation of an array into a list of integers."""
    if isinstance(s, str):  # Handle string case (where it's incorrectly stored)
        numbers = re.findall(r'\d+', s)  # Extract all numeric values
        return [int(x) for x in numbers]  # Convert to integers
    
    elif isinstance(s, np.ndarray):  # Handle NumPy array case
        return s.astype(int).tolist()
    
    elif isinstance(s, list):  # Handle lists with possible string numbers
        return [int(x) for x in s if str(x).isdigit()]
    
    return []  # Return empty list if the format is unexpected

# Apply function and debug output
playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(lambda x: convert_string_array_to_list(x))

# Check if the transformation worked
print(playlist[['tracks_to_predict']].head(10))  # Print first 10 rows

                                   tracks_to_predict
0  [218708, 242974, 165272, 9418, 221860, 229224,...
1  [232845, 111887, 144160, 10663, 216321, 138869...
2  [42986, 156208, 73950, 54114, 134077, 214967, ...
3  [166268, 22122, 211269, 71335, 21853, 190702, ...
4  [209088, 186942, 33032, 27612, 33426, 201531, ...
5  [250306, 36356, 119562, 66146, 206650, 166690,...
6  [48557, 142203, 66610, 16213, 112277, 43008, 2...
7  [45655, 100810, 250258, 169756, 86872, 117869,...
8  [88490, 122326, 87495, 205574, 3345, 191269, 2...
9  [89301, 74022, 221104, 218307, 229795, 129045,...


# Baseline Model (Random Recommendation)

In [56]:
# Set random seed for reproducibility
np.random.seed(42)

# Number of random recommendations per playlist
num_recommendations = 50

# Function to generate random recommendations only for `test` dataset type
def generate_random_recommendations(row, track_pool, num_recommendations):
    if row['dataset_type'] != 'test':  
        return []  # No recommendations for rows other than 'test'
    
    existing_tracks = set(row['track_idx_list'])
    possible_tracks = list(set(track_pool) - existing_tracks)  # Exclude existing tracks
    
    return np.random.choice(possible_tracks, num_recommendations, replace=False).tolist() if len(possible_tracks) >= num_recommendations else possible_tracks

# Get the pool of all available track indices
track_pool = track['track_idx'].tolist()

# Filter playlist to only apply the recommendation function on `test` rows
playlist['recommendations'] = playlist[playlist['dataset_type'] == 'test'].apply(
    lambda row: generate_random_recommendations(row, track_pool, num_recommendations), axis=1)

# Print results
random_recomendation = playlist.loc[playlist['dataset_type'] == 'test', ['dataset_type', 'track_idx_list', 'tracks_to_predict', 'recommendations']]
print(random_recomendation)

      dataset_type                                     track_idx_list  \
6             test  ['63526' '189533' '78677' '63380' '41539' '670...   
7             test  ['14962' '47557' '91553' '104020' '242077' '85...   
9             test  ['183059' '19373' '78643' '73807' '86276' '144...   
11            test  ['56172' '7246' '182126' '92917' '68192' '2411...   
13            test  ['191641' '200050' '174051' '205054' '90724' '...   
...            ...                                                ...   
18490         test  ['142265' '169274' '227552' '132684' '178035' ...   
18494         test  ['11910' '178754' '213722' '128015' '137053' '...   
18496         test  ['39914' '35232' '242077' '52667' '69011' '207...   
18501         test  ['140314' '14975' '99481' '234471' '72522' '18...   
18503         test  ['45959' '231436' '186479' '30051' '2180' '291...   

                                       tracks_to_predict  \
6      [48557, 142203, 66610, 16213, 112277, 43008, 2...   
7  

# Evaluate Random Model

In [59]:
def compute_metrics_for_playlist(predicted_tracks, test_indices, k):
    top_k = predicted_tracks[:k]  # Consider only top K predictions

    # Hit@K
    hit = int(any(t in top_k for t in test_indices))

    # MRR and AP calculations
    precisions = []
    num_hits = 0
    mrr = 0.0

    for rank_idx, track_idx in enumerate(top_k):
        if track_idx in test_indices:
            num_hits += 1
            precision_at_k = num_hits / (rank_idx + 1)
            precisions.append(precision_at_k)
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)

    ap = np.mean(precisions) if precisions else 0.0

    return hit, mrr, ap

def evaluate_model(playlist, k):
    test_playlists = playlist[playlist['dataset_type'] == 'test']
    num_playlists = len(test_playlists)

    hit_total, mrr_total, ap_total = 0, 0, 0

    for idx, row in test_playlists.iterrows():
        test_indices = row['tracks_to_predict']
        predicted_tracks = row['recommendations']

        hit, mrr, ap = compute_metrics_for_playlist(predicted_tracks, test_indices, k)
        
        hit_total += hit
        mrr_total += mrr
        ap_total += ap

    # Compute averages
    hit_ratio = hit_total / num_playlists if num_playlists > 0 else 0
    mrr_avg = mrr_total / num_playlists if num_playlists > 0 else 0
    map_avg = ap_total / num_playlists if num_playlists > 0 else 0

    return hit_ratio, mrr_avg, map_avg

# Run evaluation
hit_ratio, mrr_avg, map_avg = evaluate_model(random_recomendation, k=50)

# Print results
print(f"Hit@50: {hit_ratio:.4f}")
print(f"MRR: {mrr_avg:.4f}")
print(f"MAP@50: {map_avg:.4f}")

Hit@50: 0.0035
MRR: 0.0002
MAP@50: 0.0002
